# Volume + particles: composition through depth peeling

A simulation in the spirit of
[issue #277](https://github.com/K3D-tools/K3D-jupyter/issues/277): five stars
near a revolving pentagon - the symmetry deliberately broken, each star with
its own orbit radius, height, mass and wind - blowing spiral plasma arms that
feed into each other. The flow is computed in the frame co-rotating with the
stars, where the configuration is static and one velocity field covers the
whole run. Dye from the five source rings is advected semi-Lagrangian style
on a 160^3 grid for 320 steps (~2-3 minutes in pure numpy) and rendered as a
`volume`, with the stars as sphere meshes inside it.

In [ ]:
!pip install tqdm

In [6]:
import numpy as np

import k3d

In [7]:
from tqdm.auto import tqdm

rng = np.random.default_rng(277)
n = 160
L = 2.5
dt = 1.0
steps = 320

g = np.linspace(-L, L, n, dtype=np.float32)
dg = g[1] - g[0]
Zg, Yg, Xg = np.meshgrid(g, g, g, indexing='ij')

# five stars near a pentagon, symmetry deliberately broken: each has its own
# orbit radius, height, mass (wind strength and sphere size), beam sweep and
# arm count. The frame co-rotates with the configuration, so a single static
# field covers the whole run.
S = 5
ang = (np.arange(S) * 2.0 * np.pi / S + rng.uniform(-0.35, 0.35, S)).astype(np.float32)
rad = (1.05 * (1.0 + rng.uniform(-0.22, 0.22, S))).astype(np.float32)
mass = rng.uniform(0.5, 1.7, S).astype(np.float32)
sweep = rng.uniform(0.012, 0.03, S).astype(np.float32)
arms = np.where(mass == mass.max(), 2, 1)          # the heavyweight gets two arms
star_size = (0.34 * mass ** 0.33).astype(np.float32)
stars = np.stack([rad * np.cos(ang), rad * np.sin(ang),
                  rng.uniform(-0.22, 0.22, S)], axis=1).astype(np.float32)

def smooth(a, passes):
    for _ in range(passes):
        a = sum(np.roll(a, s, ax) for ax in range(3) for s in (-1, 1)) / 6.0
    return a

# the carousel: differential rotation about the mass centre winds the five
# arms around each other
bc = (stars[:, :2] * mass[:, None]).sum(axis=0) / mass.sum()
r_bc = np.sqrt((Xg - bc[0]) ** 2 + (Yg - bc[1]) ** 2) + 1e-6
omega_g = 0.034 / (0.35 + r_bc)
vx = -omega_g * (Yg - bc[1])
vy = omega_g * (Xg - bc[0])
vz = np.zeros_like(Xg)

breeze = 0.012 * np.exp(-(r_bc / 0.5) ** 2)
vx += breeze * (Xg - bc[0]) / r_bc
vy += breeze * (Yg - bc[1]) / r_bc

sources = []
thetas = []
d_min = np.full((n, n, n), np.inf, dtype=np.float32)

for i, p in enumerate(stars):
    dx = Xg - p[0]
    dy = Yg - p[1]
    dz = Zg - p[2]
    d_cyl = np.sqrt(dx ** 2 + dy ** 2) + 1e-6
    d = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)

    # each wind must reach past the mid-point to its neighbours or the arms
    # never interact; heavier stars blow farther and stir harder
    reach = np.exp(-(d / (0.8 * mass[i] ** 0.3)) ** 2)
    v_r = 0.0095 * mass[i] * np.clip(d_cyl / 0.18, 0.0, 1.0) * reach
    omega = 0.055 * mass[i] / (0.15 + d_cyl) * reach

    vx += v_r * dx / d_cyl - omega * dy
    vy += v_r * dy / d_cyl + omega * dx

    sources.append((np.exp(-((d_cyl - 0.22) / 0.035) ** 2)
                    * np.exp(-(dz / 0.04) ** 2) * mass[i]).astype(np.float32))
    thetas.append(np.arctan2(dy, dx).astype(np.float32))
    # distance in units of the star's size - the hollow scales with the sphere
    d_min = np.minimum(d_min, d / star_size[i])

A = [smooth(rng.random((n, n, n)).astype(np.float32) - 0.5, 8) for _ in range(3)]
curl_x = (np.roll(A[2], -1, 1) - np.roll(A[2], 1, 1)) - (np.roll(A[1], -1, 0) - np.roll(A[1], 1, 0))
curl_y = (np.roll(A[0], -1, 0) - np.roll(A[0], 1, 0)) - (np.roll(A[2], -1, 2) - np.roll(A[2], 1, 2))
curl_z = (np.roll(A[1], -1, 2) - np.roll(A[1], 1, 2)) - (np.roll(A[0], -1, 1) - np.roll(A[0], 1, 1))
amp = 0.5 * np.clip(d_min / 1.1, 0.0, 1.0) * np.clip(r_bc / 0.7, 0.0, 1.0)
vx += amp * curl_x
vy += amp * curl_y
vz += amp * curl_z * 0.25

# semi-Lagrangian dye advection (grid indexed [z, y, x])
def trilinear(field, fi, fj, fk):
    i0 = np.floor(fi).astype(np.int32); j0 = np.floor(fj).astype(np.int32); k0 = np.floor(fk).astype(np.int32)
    di = (fi - i0).astype(np.float32); dj = (fj - j0).astype(np.float32); dk = (fk - k0).astype(np.float32)
    i0 = np.clip(i0, 0, n - 2); j0 = np.clip(j0, 0, n - 2); k0 = np.clip(k0, 0, n - 2)
    i1 = i0 + 1; j1 = j0 + 1; k1 = k0 + 1
    return (field[i0, j0, k0] * (1 - di) * (1 - dj) * (1 - dk)
            + field[i1, j0, k0] * di * (1 - dj) * (1 - dk)
            + field[i0, j1, k0] * (1 - di) * dj * (1 - dk)
            + field[i0, j0, k1] * (1 - di) * (1 - dj) * dk
            + field[i1, j1, k0] * di * dj * (1 - dk)
            + field[i1, j0, k1] * di * (1 - dj) * dk
            + field[i0, j1, k1] * (1 - di) * dj * dk
            + field[i1, j1, k1] * di * dj * dk)

I, J, K = np.meshgrid(np.arange(n, dtype=np.float32), np.arange(n, dtype=np.float32),
                      np.arange(n, dtype=np.float32), indexing='ij')
bi = I - vz * dt / dg
bj = J - vy * dt / dg
bk = K - vx * dt / dg

# the winds cancel near the mass centre and turbulence traps dye there - the
# decay field carries a sink at the core so the centre never saturates; old
# material fades before the carousel smears it into a uniform annulus
decay = (0.995 * (1.0 - 0.09 * np.exp(-(r_bc / 0.55) ** 2))).astype(np.float32)

dye = np.zeros((n, n, n), dtype=np.float32)
for s in tqdm(range(steps), desc='advecting dye'):
    dye = trilinear(dye, bi, bj, bk)
    for i in range(S):
        mod = (0.06 + 0.94 * (0.5 + 0.5 * np.cos(arms[i] * thetas[i]
                                                 + ang[i] + sweep[i] * s)) ** 4)
        dye += sources[i] * mod.astype(np.float32) * 3.0
    dye *= decay

# hollow the stars out of the dye and compensate the radial dilution
dye *= np.clip(d_min, 0.0, 1.0) ** 1.5
dye *= (0.3 + r_bc) ** 1.2
density = (dye * 1150.0 / np.percentile(dye[dye > 0.01], 99.9)).astype(np.float32)

advecting dye:   0%|          | 0/320 [00:00<?, ?it/s]

Without depth peeling the volume cannot interleave with geometry - the
stars render entirely in front of (or behind) the plasma:

In [8]:
# on white the empty core glares like a searchlight - deep space instead
plot = k3d.plot(grid_visible=False, camera_auto_fit=False,
                background_color=0x05060D)
plot += k3d.volume(
    density, samples=512, alpha_coef=40,
    color_map=k3d.matplotlib_color_maps.jet, color_range=[50, 550],
    bounds=[-L, L, -L, L, -L, L],
)
plot += k3d.points(stars, shader='mesh', point_sizes=star_size, mesh_detail=4,
                   colors=np.array([0x2040FF, 0x3060FF, 0x30A0FF,
                                    0x2080E0, 0x5050FF], dtype=np.uint32))
plot.camera = [4.4, -4.4, 2.9, 0, 0, 0, 0, 0, 1]
plot.display()

Output()

With `depth_peels >= 3` the ray march splits into segments bounded by the
peel layers and the stars occlude and are occluded sample-accurately
(fewer layers make the segmentation too coarse to be predictable):

In [9]:
plot.depth_peels = 4

The composition is orthogonal to lighting - the advanced renderer adds
environment light and ambient occlusion on top:

In [ ]:
plot.renderer = 'advanced'
plot.environment = 'moonless_golf'

The wind that painted the arms can also be drawn directly: streamlines seeded
on the source rings follow the same velocity field the dye was advected by,
so they spiral along the very filaments they created. White lit tubes
(`lines` with segment indices); segments running through empty space are
not drawn:

In [10]:
# streamlines of the wind field, integrated with the midpoint rule and the
# same trilinear kernel the advection uses
rng_sl = np.random.default_rng(42)
per_star = 14
sl_steps = 240

def grid_at(field, p):
    fi = (p[:, 2] + L) / dg
    fj = (p[:, 1] + L) / dg
    fk = (p[:, 0] + L) / dg
    return trilinear(field, fi, fj, fk)

def wind_at(p):
    return np.column_stack([grid_at(vx, p), grid_at(vy, p), grid_at(vz, p)])

seeds = []
for i, p in enumerate(stars):
    a = rng_sl.uniform(0.0, 2.0 * np.pi, per_star)
    seeds.append(np.column_stack([
        p[0] + 0.22 * np.cos(a),
        p[1] + 0.22 * np.sin(a),
        p[2] + rng_sl.uniform(-0.05, 0.05, per_star),
    ]))
pos = np.concatenate(seeds).astype(np.float32)
P = len(pos)

dsm = density / density.max()

trail = [pos.copy()]
dens = [grid_at(dsm, pos).astype(np.float32)]
for s in range(sl_steps):
    mid = pos + 0.5 * wind_at(pos)
    pos = (pos + wind_at(mid)).astype(np.float32)
    if s % 2 == 0:
        trail.append(pos.copy())
        dens.append(grid_at(dsm, pos).astype(np.float32))

trail = np.stack(trail)                    # [T, P, 3]
dens = np.stack(dens)                      # [T, P]
T = trail.shape[0]

# tube meshes need no NaN separators: segment indices connect points only
# within a streamline, and segments through empty space stay undrawn
verts = trail.transpose(1, 0, 2).reshape(-1, 3)
attr = dens.T.reshape(-1)
tt = np.arange(T - 1)
seg = np.stack([tt, tt + 1], axis=1)
seg_all = seg[None] + (np.arange(P) * T)[:, None, None]
in_cloud = (dens[:-1] > 0.06) & (dens[1:] > 0.06)      # [T-1, P]
indices = seg_all[in_cloud.T].astype(np.uint32)

wind_lines = k3d.lines(verts, indices, indices_type='segment', shader='mesh',
                       width=0.006, color=0xFFFFFF)
plot += wind_lines